### [User Input Required] Read CSV Source with Autoloader.

__Common data loading patterns__: https://docs.databricks.com/en/ingestion/auto-loader/patterns.html

__Autoloader Options for CSV__: https://docs.databricks.com/en/ingestion/auto-loader/options.html#csv-options

__Common Autoloader Options__: https://docs.databricks.com/en/ingestion/auto-loader/options.html#common-auto-loader-options

__Schema Evolution Modes__: https://docs.databricks.com/en/ingestion/auto-loader/schema.html#how-does-auto-loader-schema-evolution-work


In [0]:
CREATE TEMPORARY STREAMING LIVE VIEW autoloader_tmp_table AS
SELECT
  *
FROM
  STREAM READ_FILES(
    'dbfs:/databricks-datasets/nyctaxi/tripdata/yellow',
    format => 'csv',
    header => 'true',
    inferSchema => 'true',
    delimiter => ',',
    schemaEvolutionMode => 'addNewColumns',
    rescuedDataColumn => '_rescued_data'
  );

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:103)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$2(SequenceExecutionState.scala:103)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$2$adapted(SequenceExecutionState.scala:100)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:100)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:720)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:439)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:439)
	at com.databricks.spark.chauffeur.ChauffeurState.cancelExecutio

### [User Input Required] Transformations + Write Data to Unity Catalog

In [0]:
CREATE
OR REFRESH STREAMING TABLE bronze_nyctaxi_tripdata_yellow COMMENT "NYC Taxi Trip Records - Yellow Taxi Trip Records" AS
SELECT
  *
EXCEPT(
    ` pickup_datetime`,
    ` dropoff_datetime`,
    ` passenger_count`,
    ` trip_distance`,
    ` pickup_longitude`,
    ` pickup_latitude`,
    ` rate_code`,
    ` store_and_fwd_flag`,
    ` dropoff_longitude`,
    ` dropoff_latitude`,
    ` payment_type`,
    ` fare_amount`,
    ` mta_tax`,
    ` tip_amount`,
    ` tolls_amount`,
    ` total_amount`,
    ` surcharge`
  )
FROM
  stream(LIVE.autoloader_tmp_table)

In [0]:
CREATE
OR REFRESH STREAMING TABLE silver_nyctaxi_tripdata_yellow COMMENT "NYC Taxi Trip Records - Yellow Taxi Trip Records" AS
SELECT
  *
FROM
  stream(LIVE.bronze_nyctaxi_tripdata_yellow)